# WISDM HAR Baseline — النسخة المصححة

## الإصلاحات الرئيسية:
1. **Subject-wise split** — فصل الأشخاص (Train/Test على مستوى المستخدم وليس الـ segments)
2. **Normalization** — تطبيع على بيانات التدريب فقط
3. **Validation set** — فصل validation من التدريب
4. **تقييم شامل** — F1-score, Precision, Recall, Confusion Matrix (وليس Accuracy فقط)
5. **منع الـ Data Leakage** — لا overlap بين train و test

## مراحل الـ Pipeline:
1. Read & Clean
2. Subject-wise Split (قبل الـ Segmentation)
3. Segmentation (train و test بشكل منفصل)
4. Normalization (fit على train فقط)
5. CNN Training
6. Evaluation شامل

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, 
                             confusion_matrix, 
                             f1_score, 
                             precision_score, 
                             recall_score, 
                             accuracy_score)
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)
torch.manual_seed(42)
print('All imports done.')

## 2. Read & Clean WISDM Dataset

WISDM v1.1 يحتوي على 6 أنشطة: Walking, Jogging, Upstairs, Downstairs, Sitting, Standing

In [ ]:
data = pd.read_csv(
    "/content/WISDM_ar_v1.1_raw.txt",
    header=None,
    names=["user", "activity", "timestamp", "x", "y", "z"],
    on_bad_lines="skip"
)

data["z"] = data["z"].astype(str).str.replace(";", "", regex=False)
data["x"] = pd.to_numeric(data["x"], errors="coerce")
data["y"] = pd.to_numeric(data["y"], errors="coerce")
data["z"] = pd.to_numeric(data["z"], errors="coerce")
data = data.dropna()

print('Total samples:', len(data))
print('Number of users:', data["user"].nunique())
print('Activities:', data["activity"].unique())
print('Activity counts:')
print(data["activity"].value_counts())

## 3. Subject-wise Split 🔒

**هذا هو الإصلاح الأهم.** بدلاً من تقسيم الـ segments عشوائياً (مما يسبب data leakage)،
نقسم على مستوى **الأشخاص**:
- Train users: 80% من الأشخاص
- Test users: 20% من الأشخاص

هذا يضمن أن النموذج لا يرى أي بيانات من أشخاص الاختبار أثناء التدريب.

كذلك نقسم train إلى train + validation (80/10/10 على مستوى الأشخاص).

In [ ]:
# كل الأشخاص
all_users = data["user"].unique()
print(f'Total users: {len(all_users)}')

# تقسيم: 80% train, 20% test (على مستوى الأشخاص)
train_users, test_users = train_test_split(
    all_users, test_size=0.20, random_state=42
)

# من train: نأخذ 10% كـ validation
train_users, val_users = train_test_split(
    train_users, test_size=0.125, random_state=42  # 0.125 * 0.80 = 0.10
)

print(f'Train users: {len(train_users)} ({len(train_users)/len(all_users)*100:.0f}%)')
print(f'Val users:   {len(val_users)} ({len(val_users)/len(all_users)*100:.0f}%)')
print(f'Test users:  {len(test_users)} ({len(test_users)/len(all_users)*100:.0f}%)')

# تقسيم البيانات حسب الأشخاص
train_data = data[data["user"].isin(train_users)].copy()
val_data   = data[data["user"].isin(val_users)].copy()
test_data  = data[data["user"].isin(test_users)].copy()

print(f'\nTrain samples: {len(train_data)}')
print(f'Val samples:   {len(val_data)}')
print(f'Test samples:  {len(test_data)}')

## 4. Segmentation

Window Size = 50, Step Size = 25, Overlap = 50%

نطبق الـ segmentation **بشكل منفصل** على train, val, test —
لكل مجموعة على حدة، ونقسم لكل user + activity لمنع النوافذ المختلطة.

In [ ]:
def create_segments(df, window_size=50, step_size=25):
    """تقسيم الإشارة إلى نوافذ، لكل user+activity بشكل مستقل."""
    segments = []
    labels = []
    
    for (user, activity), group in df.groupby(["user", "activity"]):
        group = group.reset_index(drop=True)
        for start in range(0, len(group) - window_size + 1, step_size):
            end = start + window_size
            segment = group.loc[start:end-1, ["x", "y", "z"]].values
            segments.append(segment)
            labels.append(activity)
    
    return np.array(segments), np.array(labels)

X_train_seg, y_train_seg = create_segments(train_data)
X_val_seg,   y_val_seg   = create_segments(val_data)
X_test_seg,  y_test_seg  = create_segments(test_data)

print(f'Train segments: {len(X_train_seg)}')
print(f'Val segments:   {len(X_val_seg)}')
print(f'Test segments:  {len(X_test_seg)}')
print(f'\nSegment shape: {X_train_seg[0].shape}')

## 5. Encode Labels

6 أنشطة → أرقام 0-5

In [ ]:
label_encoder = LabelEncoder()

# fit على train فقط (لمنع تسريب معلومات من test)
y_train_enc = label_encoder.fit_transform(y_train_seg)
y_val_enc   = label_encoder.transform(y_val_seg)
y_test_enc  = label_encoder.transform(y_test_seg)

print('Activities:', label_encoder.classes_)
print('Number of classes:', len(label_encoder.classes_))

## 6. Normalization 🔒

**هذا إصلاح مهم ثاني.** نطبّع البيانات باستخدام StandardScaler.

**القاعدة الذهبية:** نحسب mean و std من **التدريب فقط**، ثم نطبّقها على val و test.
لو طبّعنا على كل البيانات قبل التقسيم → تسريب معلومات من test إلى train.

In [ ]:
# Reshape: (segments, window, channels) → (segments * window, channels)
# لتطبيق StandardScaler على كل channel بشكل مستقل
n_train = len(X_train_seg)
n_val   = len(X_val_seg)
n_test  = len(X_test_seg)
window_size = 50
n_channels  = 3

# Flatten to 2D: (N * window, channels)
X_train_flat = X_train_seg.reshape(-1, n_channels)
X_val_flat   = X_val_seg.reshape(-1, n_channels)
X_test_flat  = X_test_seg.reshape(-1, n_channels)

# Fit على train فقط
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_flat)
X_val_scaled   = scaler.transform(X_val_flat)
X_test_scaled  = scaler.transform(X_test_flat)

# Reshape back: (segments, channels, window) — PyTorch format
X_train_norm = X_train_scaled.reshape(n_train, n_channels, window_size)
X_val_norm   = X_val_scaled.reshape(n_val, n_channels, window_size)
X_test_norm  = X_test_scaled.reshape(n_test, n_channels, window_size)

print('Normalization done (fit on train only).')
print(f'Train mean: {X_train_norm.mean():.4f}, std: {X_train_norm.std():.4f}')
print(f'Val   mean: {X_val_norm.mean():.4f}, std: {X_val_norm.std():.4f}')
print(f'Test  mean: {X_test_norm.mean():.4f}, std: {X_test_norm.std():.4f}')

## 7. PyTorch Tensors + DataLoaders

In [ ]:
X_train_tensor = torch.tensor(X_train_norm, dtype=torch.float32)
X_val_tensor   = torch.tensor(X_val_norm, dtype=torch.float32)
X_test_tensor  = torch.tensor(X_test_norm, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train_enc, dtype=torch.long)
y_val_tensor   = torch.tensor(y_val_enc, dtype=torch.long)
y_test_tensor  = torch.tensor(y_test_enc, dtype=torch.long)

print(f'X_train: {X_train_tensor.shape}, y_train: {y_train_tensor.shape}')
print(f'X_val:   {X_val_tensor.shape}, y_val:   {y_val_tensor.shape}')
print(f'X_test:  {X_test_tensor.shape}, y_test:  {y_test_tensor.shape}')

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset   = TensorDataset(X_val_tensor, y_val_tensor)
test_dataset  = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f'\nTrain batches: {len(train_loader)}')
print(f'Val batches:   {len(val_loader)}')
print(f'Test batches:  {len(test_loader)}')

## 8. CNN Model

نفس معمارية النموذج الأصلي (1D-CNN):
- Conv1d(3→32, k=3) → ReLU → MaxPool(2)
- Conv1d(32→64, k=3) → ReLU → GlobalAvgPool
- Linear(64→6)

In [ ]:
class HAR_CNN(nn.Module):
    def __init__(self):
        super(HAR_CNN, self).__init__()
        self.conv1 = nn.Conv1d(3, 32, kernel_size=3)
        self.relu  = nn.ReLU()
        self.pool  = nn.MaxPool1d(2)
        self.conv2 = nn.Conv1d(32, 64, kernel_size=3)
        self.gap   = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Linear(64, 6)

    def forward(self, x):
        x = self.conv1(x)
        x = self.relu(x)
        x = self.pool(x)
        x = self.conv2(x)
        x = self.relu(x)
        x = self.gap(x)
        x = x.squeeze(-1)
        x = self.classifier(x)
        return x

model = HAR_CNN()
print(model)
print(f'Total parameters: {sum(p.numel() for p in model.parameters()):,}')

## 9. Loss + Optimizer

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
num_epochs = 10

## 10. Training (with Validation)

في كل epoch نقيّم على validation set لمراقبة الـ overfitting.

In [ ]:
train_losses = []
val_losses = []
val_accuracies = []

for epoch in range(num_epochs):
    # --- Training ---
    model.train()
    total_loss = 0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_train_loss = total_loss / len(train_loader)
    train_losses.append(avg_train_loss)

    # --- Validation ---
    model.eval()
    val_loss = 0
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            val_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            val_total += y_batch.size(0)
            val_correct += (predicted == y_batch).sum().item()
    avg_val_loss = val_loss / len(val_loader)
    val_acc = 100 * val_correct / val_total
    val_losses.append(avg_val_loss)
    val_accuracies.append(val_acc)

    print(f'Epoch {epoch+1}/{num_epochs} | '
          f'Train Loss: {avg_train_loss:.4f} | '
          f'Val Loss: {avg_val_loss:.4f} | '
          f'Val Acc: {val_acc:.2f}%')

## 11. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(range(1, num_epochs+1), train_losses, 'b-', label='Train Loss')
axes[0].plot(range(1, num_epochs+1), val_losses, 'r-', label='Val Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training & Validation Loss')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(range(1, num_epochs+1), val_accuracies, 'g-', label='Val Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Validation Accuracy')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Final Test Evaluation

تقييم شامل على test set:
- Accuracy
- Precision (per-class)
- Recall (per-class)
- F1-score (per-class + macro/weighted)
- Confusion Matrix

In [ ]:
model.eval()
all_preds = []
all_true = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        outputs = model(X_batch)
        _, predicted = torch.max(outputs, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_true.extend(y_batch.cpu().numpy())

all_preds = np.array(all_preds)
all_true = np.array(all_true)

# Accuracy
test_acc = accuracy_score(all_true, all_preds)
print(f'Test Accuracy: {test_acc*100:.2f}%')
print()

# Classification Report (Precision, Recall, F1 per class)
print('Classification Report:')
print(classification_report(all_true, all_preds,
      target_names=label_encoder.classes_))

# Macro and Weighted averages
f1_macro    = f1_score(all_true, all_preds, average='macro')
f1_weighted = f1_score(all_true, all_preds, average='weighted')
prec_macro  = precision_score(all_true, all_preds, average='macro')
rec_macro   = recall_score(all_true, all_preds, average='macro')
print(f'Macro F1:    {f1_macro:.4f}')
print(f'Weighted F1:{f1_weighted:.4f}')
print(f'Macro Precision: {prec_macro:.4f}')
print(f'Macro Recall:    {rec_macro:.4f}')

## 13. Confusion Matrix

In [ ]:
cm = confusion_matrix(all_true, all_preds)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_encoder.classes_,
            yticklabels=label_encoder.classes_)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 14. مقارنة: قبل الإصلاح vs بعد الإصلاح

| | قبل (Data Leakage) | بعد (Subject Split) |
|---|---|---|
| Split | عشوائي على segments | فصل الأشخاص |
| Overlap بين train/test | ✅ موجود (نصف النافذة مشترك) | ❌ ممنوع |
| Normalization | غير موجود | StandardScaler (fit على train) |
| Validation | غير موجود | Subject-wise val set |
| Metrics | Accuracy فقط | Accuracy + F1 + Precision + Recall + Confusion Matrix |
| Accuracy المتوقع | ~96% (مضخّم) | ~75-85% (حقيقي) |

**الدقة الحقيقية ستهبط — وهذا طبيعي وصحي.**
الدقة العالية السابقة كانت وهمية بسبب الـ data leakage.